In [1]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:86% !important;}
div.cell.code_cell.rendered{width:100%;}
div.CodeMirror {font-family:Consolas; font-size:15pt;}
div.output {font-size:15pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:15pt;}
div.prompt {min-width:70px;
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:12pt;padding:5px;}
table.dataframe{font-size:15px;}
</style>
"""))

# 인코더 LSTM 과 디코더 LSTM(Sequence to Sequence)로 번역하기

## 1. 패키지 impiort & 하이퍼파라미터
- 하이퍼마라미터 : 모델의 정확도 및 학습속도에 영향을 미치는 변수

In [2]:
import numpy as np
import pandas as pd
from time import time

from tensorflow.keras.layers import Input,LSTM, Dense
from tensorflow.keras.models import Model
from  tensorflow.keras.utils import to_categorical

# 하이퍼파라미터
MY_HIDDEN = 128
MY_EPOCH = 500

## 2. 번역데이터 불러오기


In [6]:
raw = pd.read_csv('data/translate.csv', header=None)
eng_kor = raw.values.tolist() # 데이터프레임을 리스트로 변환
print('번역데이터 수 :', len(eng_kor))

번역데이터 수 : 110


## 3. 영어알파벳과 한글문자 리스트 만들기

In [24]:
e_alpha = [c for c in 'SEPabcdefghijklmnopqrstuvwexyz']
e_alpha
{c:i for i, c in enumerate(e_alpha)}
korean = ''.join([data[1] for data in eng_kor])
print(set([ch for ch in korean]))
k_ch = list(set([ch for ch in korean]))
k_ch.sort()


{'책', '파', '상', '짜', '류', '사', '램', '남', '아', '관', '반', '옥', '비', '먼', '가', '을', '구', '통', '입', '은', '인', '계', '얼', '왼', '그', '우', '머', '탈', '붕', '소', '회', '칙', '단', '험', '쪽', '것', '도', '유', '주', '지', '손', '다', '색', '흐', '키', '쉽', '고', '서', '읽', '방', '뿌', '복', '미', '늦', '메', '실', '적', '이', '익', '연', '리', '자', '위', '획', '행', '오', '번', '찾', '피', '간', '굴', '물', '날', '선', '좋', '운', '름', '핑', '노', '얇', '거', '목', '약', '의', '합', '게', '모', '내', '규', '래', '망', '한', '각', '바', '많', '깊', '식', '음', '광', '시', '용', '작', '농', '생', '나', '감', '휴', '뉴', '들', '장', '수', '부', '스', '싸', '람', '언', '편', '개', '택', '분', '랑', '어', '넓', '놀', '높', '여', '움', '금', '제', '요', '팔', '멍', '녀', '매', '출', '릎', '동', '무', '해', '크', '기', '명'}


In [31]:
k_alpha = pd.read_csv('data/korean.csv', header=None)[0].tolist()
k_alpha == k_ch  # 순서와 내용이 모두 같음

True

In [34]:
from collections import Counter
list1 = ['가', '간', '나']
list2 = ['간', '나', '가']
list1 == list2
Counter(list1) == Counter(list2)

True

In [35]:
alpha = e_alpha + k_alpha
print(alpha)
alpha_total_size = len(alpha)
print(alpha_total_size)

['S', 'E', 'P', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'e', 'x', 'y', 'z', '가', '각', '간', '감', '개', '거', '것', '게', '계', '고', '관', '광', '구', '굴', '규', '그', '금', '기', '깊', '나', '날', '남', '내', '넓', '녀', '노', '놀', '농', '높', '뉴', '늦', '다', '단', '도', '동', '들', '람', '랑', '래', '램', '류', '름', '릎', '리', '많', '망', '매', '머', '먼', '멍', '메', '명', '모', '목', '무', '물', '미', '바', '반', '방', '번', '복', '부', '분', '붕', '비', '뿌', '사', '상', '색', '생', '서', '선', '소', '손', '수', '쉽', '스', '시', '식', '실', '싸', '아', '약', '얇', '어', '언', '얼', '여', '연', '오', '옥', '왼', '요', '용', '우', '운', '움', '위', '유', '은', '을', '음', '의', '이', '익', '인', '읽', '입', '자', '작', '장', '적', '제', '좋', '주', '지', '짜', '쪽', '찾', '책', '출', '칙', '크', '키', '탈', '택', '통', '파', '팔', '편', '피', '핑', '한', '합', '해', '행', '험', '회', '획', '휴', '흐']
172


## 4. 문자당 num를 갖는 dict 만들기

In [36]:
char_to_num = {}
for i, c in enumerate(alpha):
    char_to_num[c] = i

In [38]:
char_to_num = {c : i for i, c in enumerate(alpha)}
print(char_to_num)

{'S': 0, 'E': 1, 'P': 2, 'a': 3, 'b': 4, 'c': 5, 'd': 6, 'e': 26, 'f': 8, 'g': 9, 'h': 10, 'i': 11, 'j': 12, 'k': 13, 'l': 14, 'm': 15, 'n': 16, 'o': 17, 'p': 18, 'q': 19, 'r': 20, 's': 21, 't': 22, 'u': 23, 'v': 24, 'w': 25, 'x': 27, 'y': 28, 'z': 29, '가': 30, '각': 31, '간': 32, '감': 33, '개': 34, '거': 35, '것': 36, '게': 37, '계': 38, '고': 39, '관': 40, '광': 41, '구': 42, '굴': 43, '규': 44, '그': 45, '금': 46, '기': 47, '깊': 48, '나': 49, '날': 50, '남': 51, '내': 52, '넓': 53, '녀': 54, '노': 55, '놀': 56, '농': 57, '높': 58, '뉴': 59, '늦': 60, '다': 61, '단': 62, '도': 63, '동': 64, '들': 65, '람': 66, '랑': 67, '래': 68, '램': 69, '류': 70, '름': 71, '릎': 72, '리': 73, '많': 74, '망': 75, '매': 76, '머': 77, '먼': 78, '멍': 79, '메': 80, '명': 81, '모': 82, '목': 83, '무': 84, '물': 85, '미': 86, '바': 87, '반': 88, '방': 89, '번': 90, '복': 91, '부': 92, '분': 93, '붕': 94, '비': 95, '뿌': 96, '사': 97, '상': 98, '색': 99, '생': 100, '서': 101, '선': 102, '소': 103, '손': 104, '수': 105, '쉽': 106, '스': 107, '시': 108, '식': 109, '실': 110, '싸': 11

In [42]:
data= eng_kor[0]
print(data)
print(char_to_num['c'],char_to_num['o'],char_to_num['l'],char_to_num['d'])
print('인코더 입력(원핫인코딩전) ',[char_to_num[c] for c in data[0]])
print('디코더 입력(원핫인코딩전) : ',[char_to_num[c] for c in 'S'+data[1]])
print('디코더 출력(원핫인코딩x)', [char_to_num[c] for c in data[1]+'E'])

['cold', '감기']
5 17 14 6
[5, 17, 14, 6]
디코더 입력(원핫인코딩전) :  [0, 33, 47]
디코더 출력(원핫인코딩x) [33, 47, 1]


In [ ]:
# 희소행렬의 원핫인코딩방법1  주의 get dummy 를 쓰면 안됨

In [43]:
to_categorical([5,7,6,8], num_classes=10)

array([[0., 0., 0., 0., 0., 1., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 1., 0., 0.],
       [0., 0., 0., 0., 0., 0., 1., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 1., 0.]], dtype=float32)

In [45]:
# 희소 행렬의 원핫인코딩방법2
np.eye(10)

array([[1., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 1., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 1., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 1., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 1., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 1., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 1., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 1., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 1., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 1.]])

In [60]:
def encoding(eng_kor=eng_kor):
    enc_in = [] # 인코더 입력
    dec_in = [] # 디코더 입력
    dec_out = [] # 디코더 출력(타겟)
    for data in eng_kor:
        # 인코더 입력데이터(영어알파벳 -> 숫자 -> 원핫인코딩)
        eng = [char_to_num[c] for c in data[0]]
        eng_one = np.eye(alpha_total_size)[eng]
        # print('영어 :', eng, eng_one)
        enc_in.append(eng_one) # eng_one의 shape : (4,171)
        
        # 디코더 입력데이터("S한글" -> 숫자 -> 원핫인코딩)
        kor = [char_to_num[c] for c in "S"+data[1]]
        #kor_one = to_categorical(kor, num_classes=alpha_total_size)
        kor_one = np.eye(alpha_total_size)[kor] # kor_one의 shape : (3, 171)
        # print('한글 :', kor, kor_one)
        dec_in.append(kor_one)
        
        # 디코더 출력데이터("한글E" -> 숫자)
        kor = [char_to_num[c] for c in data[1]+"E"]
        # print(kor)
        dec_out.append(kor)
    return enc_in, dec_in, dec_out

In [61]:
sample = [['cold', '감기'], ['wood','나무']]
x_enc, x_dec, y_dec = encoding(sample)
X_enc = np.array(x_enc)
X_dec = np.array(x_dec)
Y_dec = np.array(y_dec)
X_enc.shape, X_dec.shape, Y_dec.shape # (2,3,1)

((2, 4, 172), (2, 3, 172), (2, 3))

In [62]:
# 축 증가 방법 : 맨 바지막축 증가하는 함수
np.expand_dims(Y_dec, axis=1)

array([[[33, 47,  1]],

       [[49, 84,  1]]])

In [63]:
# 축 증가 방법3
Y_dec[..., np.newaxis]

array([[[33],
        [47],
        [ 1]],

       [[49],
        [84],
        [ 1]]])

In [64]:
# 축 증가 방법4
Y_dec[:,:, None]

array([[[33],
        [47],
        [ 1]],

       [[49],
        [84],
        [ 1]]])

# 6. 전체 입력데이터, 타겟데이터 준비

In [84]:
x_enc, x_dec, y_dec = encoding(eng_kor)
X_enc = np.array(x_enc)
X_dec = np.array(x_dec)
Y_dec = np.expand_dims(y_dec, axis=-1)
#Y_dec = np.array(y_dec[..., np.newaxis])
X_enc.shape, X_dec.shape, Y_dec.shape

((110, 4, 172), (110, 3, 172), (110, 3, 1))

## 7. 모델 구현

In [85]:
# 인코더 LSTM
enc_in = Input(shape=(4, alpha_total_size))
LSTM(unit)

NameError: name 'unit' is not defined

In [86]:
# 인코더 LSTM
ENC_IN = Input(shape=(4, alpha_total_size)) # alpha_total_size:171

_, state_h, state_c = LSTM(units=MY_HIDDEN, # MY_HIDDEN:128
                           return_state=True, # return_state=True h값과 c 받기
                           # return_sequences=False # LSTM윗 출력 안 받음
                          )(ENC_IN) 

# 인코더와 디코더 연결 고리
LINK = [state_h, state_c]

# 디코더 LSTM
DEC_IN = Input(shape=(3, alpha_total_size))
DEC_MID = LSTM(units=MY_HIDDEN, # 128
              # return_state=False,
              return_sequences=True, # 윗 출력 받음
              )(DEC_IN,
               initial_state=LINK)

# 최종 출력층
DEC_OUT = Dense(units=alpha_total_size,
               activation='softmax')(DEC_MID)
# 모델
model = Model(inputs=[ENC_IN, DEC_IN],
             outputs=DEC_OUT)
model.summary()

Model: "model_3"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 input_9 (InputLayer)           [(None, 4, 172)]     0           []                               
                                                                                                  
 input_10 (InputLayer)          [(None, 3, 172)]     0           []                               
                                                                                                  
 lstm_7 (LSTM)                  [(None, 128),        154112      ['input_9[0][0]']                
                                 (None, 128),                                                     
                                 (None, 128)]                                                     
                                                                                            

In [87]:
model.compile(loss='sparse_categorical_crossentropy',
             optimizer='rmsprop',
             metrics=['accuracy'] # loss만 로그 출력
             )
begin = time()
model.fit([X_enc, X_dec], Y_dec,
         epochs=MY_EPOCH,
         verbose=1)
end = time()
print('학습시간 :', end-begin)

Epoch 1/500
4/4 [==============================] - 3s 12ms/step - loss: 5.1203 - accuracy: 0.2182
Epoch 2/500
4/4 [==============================] - 0s 11ms/step - loss: 4.9434 - accuracy: 0.3333
Epoch 3/500
4/4 [==============================] - 0s 10ms/step - loss: 4.0668 - accuracy: 0.3333
Epoch 4/500
4/4 [==============================] - 0s 10ms/step - loss: 3.4765 - accuracy: 0.3333
Epoch 5/500
4/4 [==============================] - 0s 11ms/step - loss: 3.4231 - accuracy: 0.3333
Epoch 6/500
4/4 [==============================] - 0s 10ms/step - loss: 3.3838 - accuracy: 0.3333
Epoch 7/500
4/4 [==============================] - 0s 10ms/step - loss: 3.3489 - accuracy: 0.3333
Epoch 8/500
4/4 [==============================] - 0s 10ms/step - loss: 3.3213 - accuracy: 0.3333
Epoch 9/500
4/4 [==============================] - 0s 11ms/step - loss: 3.3019 - accuracy: 0.3333
Epoch 10/500
4/4 [==============================] - 0s 10ms/step - loss: 3.2696 - accuracy: 0.3333
Epoch 11/500
4/4 [=

4/4 [==============================] - 0s 9ms/step - loss: 1.1246 - accuracy: 0.8667
Epoch 85/500
4/4 [==============================] - 0s 9ms/step - loss: 1.1023 - accuracy: 0.8758
Epoch 86/500
4/4 [==============================] - 0s 9ms/step - loss: 1.0593 - accuracy: 0.8879
Epoch 87/500
4/4 [==============================] - 0s 10ms/step - loss: 1.0329 - accuracy: 0.8879
Epoch 88/500
4/4 [==============================] - 0s 10ms/step - loss: 1.0048 - accuracy: 0.9061
Epoch 89/500
4/4 [==============================] - 0s 9ms/step - loss: 0.9876 - accuracy: 0.9000
Epoch 90/500
4/4 [==============================] - 0s 10ms/step - loss: 0.9464 - accuracy: 0.9152
Epoch 91/500
4/4 [==============================] - 0s 9ms/step - loss: 0.9187 - accuracy: 0.9212
Epoch 92/500
4/4 [==============================] - 0s 9ms/step - loss: 0.8914 - accuracy: 0.9273
Epoch 93/500
4/4 [==============================] - 0s 9ms/step - loss: 0.8462 - accuracy: 0.9303
Epoch 94/500
4/4 [============

4/4 [==============================] - 0s 8ms/step - loss: 0.0353 - accuracy: 1.0000
Epoch 167/500
4/4 [==============================] - 0s 8ms/step - loss: 0.0354 - accuracy: 1.0000
Epoch 168/500
4/4 [==============================] - 0s 8ms/step - loss: 0.0341 - accuracy: 1.0000
Epoch 169/500
4/4 [==============================] - 0s 8ms/step - loss: 0.0330 - accuracy: 1.0000
Epoch 170/500
4/4 [==============================] - 0s 8ms/step - loss: 0.0300 - accuracy: 1.0000
Epoch 171/500
4/4 [==============================] - 0s 8ms/step - loss: 0.0270 - accuracy: 1.0000
Epoch 172/500
4/4 [==============================] - 0s 8ms/step - loss: 0.0261 - accuracy: 1.0000
Epoch 173/500
4/4 [==============================] - 0s 8ms/step - loss: 0.0264 - accuracy: 1.0000
Epoch 174/500
4/4 [==============================] - 0s 8ms/step - loss: 0.0252 - accuracy: 1.0000
Epoch 175/500
4/4 [==============================] - 0s 9ms/step - loss: 0.0267 - accuracy: 1.0000
Epoch 176/500
4/4 [=====

4/4 [==============================] - 0s 10ms/step - loss: 3.5809e-04 - accuracy: 1.0000
Epoch 248/500
4/4 [==============================] - 0s 10ms/step - loss: 3.3482e-04 - accuracy: 1.0000
Epoch 249/500
4/4 [==============================] - 0s 10ms/step - loss: 3.1886e-04 - accuracy: 1.0000
Epoch 250/500
4/4 [==============================] - 0s 11ms/step - loss: 3.0020e-04 - accuracy: 1.0000
Epoch 251/500
4/4 [==============================] - 0s 11ms/step - loss: 2.8659e-04 - accuracy: 1.0000
Epoch 252/500
4/4 [==============================] - 0s 9ms/step - loss: 2.7840e-04 - accuracy: 1.0000
Epoch 253/500
4/4 [==============================] - 0s 8ms/step - loss: 2.5625e-04 - accuracy: 1.0000
Epoch 254/500
4/4 [==============================] - 0s 8ms/step - loss: 2.4046e-04 - accuracy: 1.0000
Epoch 255/500
4/4 [==============================] - 0s 8ms/step - loss: 2.2320e-04 - accuracy: 1.0000
Epoch 256/500
4/4 [==============================] - 0s 8ms/step - loss: 2.4240e-0

4/4 [==============================] - 0s 9ms/step - loss: 7.1337e-06 - accuracy: 1.0000
Epoch 327/500
4/4 [==============================] - 0s 8ms/step - loss: 6.8361e-06 - accuracy: 1.0000
Epoch 328/500
4/4 [==============================] - 0s 9ms/step - loss: 6.6410e-06 - accuracy: 1.0000
Epoch 329/500
4/4 [==============================] - 0s 8ms/step - loss: 6.5099e-06 - accuracy: 1.0000
Epoch 330/500
4/4 [==============================] - 0s 8ms/step - loss: 6.1306e-06 - accuracy: 1.0000
Epoch 331/500
4/4 [==============================] - 0s 8ms/step - loss: 5.9395e-06 - accuracy: 1.0000
Epoch 332/500
4/4 [==============================] - 0s 8ms/step - loss: 5.9409e-06 - accuracy: 1.0000
Epoch 333/500
4/4 [==============================] - 0s 9ms/step - loss: 5.6451e-06 - accuracy: 1.0000
Epoch 334/500
4/4 [==============================] - 0s 8ms/step - loss: 5.5251e-06 - accuracy: 1.0000
Epoch 335/500
4/4 [==============================] - 0s 9ms/step - loss: 5.2658e-06 - a

4/4 [==============================] - 0s 8ms/step - loss: 1.2878e-06 - accuracy: 1.0000
Epoch 406/500
4/4 [==============================] - 0s 8ms/step - loss: 1.2766e-06 - accuracy: 1.0000
Epoch 407/500
4/4 [==============================] - 0s 8ms/step - loss: 1.2582e-06 - accuracy: 1.0000
Epoch 408/500
4/4 [==============================] - 0s 8ms/step - loss: 1.2419e-06 - accuracy: 1.0000
Epoch 409/500
4/4 [==============================] - 0s 8ms/step - loss: 1.2347e-06 - accuracy: 1.0000
Epoch 410/500
4/4 [==============================] - 0s 9ms/step - loss: 1.2206e-06 - accuracy: 1.0000
Epoch 411/500
4/4 [==============================] - 0s 9ms/step - loss: 1.2055e-06 - accuracy: 1.0000
Epoch 412/500
4/4 [==============================] - 0s 10ms/step - loss: 1.1870e-06 - accuracy: 1.0000
Epoch 413/500
4/4 [==============================] - 0s 9ms/step - loss: 1.1766e-06 - accuracy: 1.0000
Epoch 414/500
4/4 [==============================] - 0s 11ms/step - loss: 1.1664e-06 -

4/4 [==============================] - 0s 8ms/step - loss: 6.5637e-07 - accuracy: 1.0000
Epoch 485/500
4/4 [==============================] - 0s 8ms/step - loss: 6.5457e-07 - accuracy: 1.0000
Epoch 486/500
4/4 [==============================] - 0s 8ms/step - loss: 6.5421e-07 - accuracy: 1.0000
Epoch 487/500
4/4 [==============================] - 0s 8ms/step - loss: 6.4698e-07 - accuracy: 1.0000
Epoch 488/500
4/4 [==============================] - 0s 8ms/step - loss: 6.4120e-07 - accuracy: 1.0000
Epoch 489/500
4/4 [==============================] - 0s 8ms/step - loss: 6.3687e-07 - accuracy: 1.0000
Epoch 490/500
4/4 [==============================] - 0s 8ms/step - loss: 6.3470e-07 - accuracy: 1.0000
Epoch 491/500
4/4 [==============================] - 0s 8ms/step - loss: 6.3289e-07 - accuracy: 1.0000
Epoch 492/500
4/4 [==============================] - 0s 8ms/step - loss: 6.2242e-07 - accuracy: 1.0000
Epoch 493/500
4/4 [==============================] - 0s 9ms/step - loss: 6.2206e-07 - a

In [83]:
print("X_enc.shape:", X_enc.shape)
print("X_dec.shape:", X_dec.shape)
print("Y_dec.shape:", Y_dec.shape)


X_enc.shape: (2, 4, 172)
X_dec.shape: (110, 3, 172)
Y_dec.shape: (110, 3, 1)


In [89]:
# 쉬운문제
easy_test=[['cold', 'PP'],
           ['fact', 'PP'],
           ['love', 'PP'],
           ['luck', 'PP'],
           ['milk', 'PP']]
enc_in, dec_in, _ = encoding(easy_test)
enc_in = np.array(enc_in)
dec_in = np.array(dec_in)
enc_in.shape, dec_in.shape

((5, 4, 172), (5, 3, 172))

In [90]:
# 위의 문제 예측

# 위의 문제 예측하기
pred = model.predict([enc_in, dec_in])
pred.argmax(axis=-1)

1/1 [==============================] - 1s 638ms/step


array([[ 33,  47,   1],
       [ 97, 110,   1],
       [ 97,  67,   1],
       [166, 126,   1],
       [125, 129,   1]], dtype=int64)

In [91]:
char_to_num['감'],alpha[32]

(33, '간')

In [94]:
for test, yhat in zip(easy_test, pred):
#     print(test[0], yhat.argmax(axis=-1))
    eng = test[0]
    hat = np.argmax(yhat, axis=-1)
    kor = ''.join([alpha[h] for h in hat[:-1]])
    print("{} => {}".format(eng, kor))

cold => 감기
fact => 사실
love => 사랑
luck => 행운
milk => 우유


In [95]:
# 어려운 문제
hard_test=[['lvoe', 'PP'],
           ['loev', 'PP'],
           ['love', 'PP'],
           ['olve', 'PP'],
           ['eovl', 'PP']]
enc_in, dec_in, _ = encoding(hard_test)
enc_in = np.array(enc_in)
dec_in = np.array(dec_in)